In [5]:
import pandas as pd
import numpy as np
import warnings
import os
from datetime import datetime
import io

warnings.filterwarnings("ignore")

log_lines = []

def log(msg):
    """Print message and store it for log file."""
    print(msg)
    log_lines.append(msg)

# ----------------------------------------
# 1. Data Acquisition
# ----------------------------------------
PATH = "vgchartz-2024.csv"
df = pd.read_csv(PATH)

raw_shape = df.shape

log("=" * 60)
log(f"PREPROCESSING LOG ")
log("=" * 60)
log(f"\n[Step 1] Raw data loaded from: {PATH}")
log(f"  Initial shape: {raw_shape[0]} rows, {raw_shape[1]} columns")
log(f"  Columns: {list(df.columns)}")

# ----------------------------------------
# 2. Data Cleaning
# ----------------------------------------

# --- 2.1 Column selection ---
cols = [
    'console', 'genre', 'publisher', 'developer', 'critic_score',
    'total_sales', 'na_sales', 'jp_sales', 'pal_sales', 'other_sales'
]
# Only keep necessary columns (img column is dropped)
df = df[cols].copy()

log("\n[Step 2] Column selection")
log(f"  Retained columns: {cols}")
log("  Justification: 'img' column not relevant for analysis; other metadata columns excluded.")
log(f"  Shape after selection: {df.shape}")

# --- 2.2 Handle missing values ---
before_dropna = df.shape[0]

# Remove rows where console or genre is missing
df = df.dropna(subset=['console', 'genre'])
after_dropna_subset = df.shape[0]

log("\n[Step 3] Missing value handling - row deletion")
log(f"  Rows before: {before_dropna}")
log(f"  Rows after removing missing console/genre: {after_dropna_subset}")
log("  Justification: 'console' and 'genre' are primary categorical variables; "
    "they cannot be reliably imputed. Removing these rows avoids misleading groupings.")

# Impute missing critic_score with median 
critic_median = df['critic_score'].median()
df['critic_score'] = df['critic_score'].fillna(critic_median)

log(f"\n  Missing 'critic_score' filled with median: {critic_median:.2f}")
log("  Justification: critic_score distribution is roughly symmetric; median is robust to outliers.")

# --- 2.3 Business-rule filtering ---
before_filter = df.shape[0]
df = df.query('total_sales > 0')
after_filter = df.shape[0]

log("\n[Step 4] Filtering: total_sales > 0")
log(f"  Rows before: {before_filter}")
log(f"  Rows after: {after_filter}")
log("  Justification: Zero or negative sales values are likely data errors or placeholders; "
    "they have no analytical value.")

# --- 2.4 Text normalization ---
df['console'] = df['console'].str.strip().str.upper()
df['genre'] = df['genre'].str.strip().str.upper()

log("\n[Step 5] Text normalization")
log("  'console' and 'genre' converted to uppercase and stripped of whitespace.")
log("  Justification: Prevents duplicate categories due to inconsistent casing or leading/trailing spaces.")

# --- 2.5 Target transformation ---
df['log_sales'] = np.log1p(df['total_sales'])

log("\n[Step 6] Derived variable: log_sales = log1p(total_sales)")
log("  Justification: Stabilises variance for regression modelling; log1p handles zero gracefully, "
    "though zeros were already filtered out.")

# ----------------------------------------
# 3. Data Quality Summary
# ----------------------------------------
log("\n" + "=" * 60)
log("DATA QUALITY SUMMARY")
log("=" * 60)

# 3.1 Null rates
log("\n1. Null rates (final dataset):")
null_rates = (df.isnull().sum() / len(df)) * 100
for col in df.columns:
    log(f"   {col}: {null_rates[col]:.2f}%")

# 3.2 Class balance for categorical variables
log("\n2. Class balance - console:")
console_counts = df['console'].value_counts()
log(console_counts.to_string())

log("\n   Class balance - genre:")
genre_counts = df['genre'].value_counts()
log(genre_counts.to_string())

# 3.3 Outlier treatment
log("\n3. Outlier treatment:")
log("   - total_sales: removed values <= 0; no capping/winsorizing applied "
    "(log transformation mitigates skew).")
log("   - critic_score: missing values imputed with median; range 0-100 confirmed reasonable.")
log("   - Regional sales (na_sales, jp_sales, pal_sales, other_sales): retained as-is; "
    "further outlier analysis may be performed during EDA.")

# 3.4 Schema / data types
log("\n4. Final schema:")
buffer = io.StringIO()
df.info(buf=buffer)
log(buffer.getvalue())

log("Data types overview:")
log(df.dtypes.to_string())

# ----------------------------------------
# 4. Save cleaned dataset and log
# ----------------------------------------
cleaned_path = "cleaned_vgchartz.csv"
df.to_csv(cleaned_path, index=False)
log(f"\n[Output] Cleaned dataset saved to: {cleaned_path}")

log_path = "preprocessing_log.txt"
with open(log_path, "w", encoding="utf-8") as f:
    f.write("\n".join(log_lines))
log(f"[Output] Preprocessing log saved to: {log_path}")

log("\n" + "=" * 60)
log("Preprocessing complete.")
log("=" * 60)

PREPROCESSING LOG 

[Step 1] Raw data loaded from: vgchartz-2024.csv
  Initial shape: 64016 rows, 14 columns
  Columns: ['img', 'title', 'console', 'genre', 'publisher', 'developer', 'critic_score', 'total_sales', 'na_sales', 'jp_sales', 'pal_sales', 'other_sales', 'release_date', 'last_update']

[Step 2] Column selection
  Retained columns: ['console', 'genre', 'publisher', 'developer', 'critic_score', 'total_sales', 'na_sales', 'jp_sales', 'pal_sales', 'other_sales']
  Justification: 'img' column not relevant for analysis; other metadata columns excluded.
  Shape after selection: (64016, 10)

[Step 3] Missing value handling - row deletion
  Rows before: 64016
  Rows after removing missing console/genre: 64016
  Justification: 'console' and 'genre' are primary categorical variables; they cannot be reliably imputed. Removing these rows avoids misleading groupings.

  Missing 'critic_score' filled with median: 7.50
  Justification: critic_score distribution is roughly symmetric; median 